# Example 12.1.1 / 1: Trace the Front with Two Sweeps

Companion notebook for **12.1.1 Pareto Fronts** in the AMPL lab Documentation, section "Trace the Front with Two Sweeps". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Trace the Front with Two Sweeps

A **weighted-sum sweep** minimizes `lambda * ScaledCost + (1 - lambda) * ScaledImpact` for a grid
of `lambda` values. It recovers supported efficient points only, and evenly spaced weights rarely
produce evenly spaced points. An **epsilon-constraint sweep** minimizes cost alone subject to
`TotalImpact <= impact_limit` for a grid of limits; because it bounds one objective instead of
weighting it, it can also recover unsupported points.

The model holds both objectives. `ScaledCost` and `ScaledImpact` map the extremes found by the
single-objective solves (cost 1 to 8 and impact 1 to 8) onto 0 to 1. The weighted run drops
`Cost` with `drop()`, assigns `lambda` by item assignment, and reads the choice with `to_dict()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set OPTIONS;                          # candidate options
param cost {OPTIONS} >= 0;            # dollars
param impact {OPTIONS} >= 0;          # tonnes
param cost_min;                       # cost scaling offset, dollars
param cost_range > 0;                 # cost scaling range, dollars
param impact_min;                     # impact scaling offset, tonnes
param impact_range > 0;               # impact scaling range, tonnes
param lambda >= 0, <= 1 default 0.5;  # weight on scaled cost in Blend
param impact_limit default Infinity;  # epsilon bound on impact, tonnes

var Pick {OPTIONS} binary;            # 1 if the option is chosen
var TotalCost = sum {o in OPTIONS} cost[o] * Pick[o];  # dollars
var TotalImpact = sum {o in OPTIONS} impact[o] * Pick[o];  # tonnes
var ScaledCost = (TotalCost - cost_min) / cost_range;  # unitless, comparable to ScaledImpact
var ScaledImpact = (TotalImpact - impact_min) / impact_range;  # unitless, comparable to ScaledCost

minimize Blend:                       # weighted-sum sweep objective
    lambda * ScaledCost + (1 - lambda) * ScaledImpact;
minimize Cost: TotalCost;             # epsilon-constraint sweep objective

subject to ChooseOne: sum {o in OPTIONS} Pick[o] = 1;  # choose exactly one option
subject to ImpactBudget: TotalImpact <= impact_limit;  # epsilon constraint on impact

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["OPTIONS"] = ["A", "B", "C", "D"]
ampl.param["cost"] = {"A": 1, "B": 4, "C": 8, "D": 5}
ampl.param["impact"] = {"A": 8, "B": 6, "C": 1, "D": 7}
ampl.param["cost_min"] = 1
ampl.param["cost_range"] = 7
ampl.param["impact_min"] = 1
ampl.param["impact_range"] = 7

In [5]:
# run.py
ampl.obj["Cost"].drop()
for weight in [0, 0.2, 0.4, 0.6, 0.8, 1]:
    ampl.param["lambda"] = weight
    ampl.solve(solver="highs", verbose=False)
    chosen = [o for o, v in ampl.var["Pick"].to_dict().items() if v > 0.5]
    print(weight, ampl.solve_result, chosen,
          ampl.var["TotalCost"].value(), ampl.var["TotalImpact"].value())

0 solved ['C'] 8.0 1.0
0.2 solved ['C'] 8.0 1.0
0.4 solved ['C'] 8.0 1.0
0.6 solved ['A'] 1.0 8.0
0.8 solved ['A'] 1.0 8.0
1 solved ['A'] 1.0 8.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set OPTIONS := A B C D;
param: cost impact :=
    A   1   8
    B   4   6
    C   8   1
    D   5   7 ;
param cost_min := 1;
param cost_range := 7;
param impact_min := 1;
param impact_range := 7;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
objective Blend;
for {weight in {0, 0.2, 0.4, 0.6, 0.8, 1}} {
    let lambda := weight;
    solve;
    printf {o in OPTIONS: Pick[o] > 0.5} "%g %s %s %g %g\n",
        weight, solve_result, o, TotalCost, TotalImpact;
}

0 solved C 8 1
0.2 solved C 8 1
0.4 solved C 8 1
0.6 solved A 1 8
0.8 solved A 1 8
1 solved A 1 8


### Another way in amplpy

Every weight returns `A` or `C`; the unsupported point `B` never appears, and the switch happens
between 0.4 and 0.6, so equal spacing in `lambda` does not give equal spacing on the front.

The epsilon-constraint sweep runs the same model with other commands. It drops `Blend`, keeps a
record per limit with `ampl.solve_result`, and stores objective values and the gap reported by
`mip:return_gap=1` only for solved runs:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["OPTIONS"] = ["A", "B", "C", "D"]
ampl.param["cost"] = {"A": 1, "B": 4, "C": 8, "D": 5}
ampl.param["impact"] = {"A": 8, "B": 6, "C": 1, "D": 7}
ampl.param["cost_min"] = 1
ampl.param["cost_range"] = 7
ampl.param["impact_min"] = 1
ampl.param["impact_range"] = 7

In [11]:
# run.py
ampl.obj["Blend"].drop()
records = []
for limit in [8, 7, 6, 5, 1, 0]:
    ampl.param["impact_limit"] = limit
    ampl.solve(solver="highs", highs_options="mip:return_gap=1", verbose=False)
    record = {"limit": limit, "status": ampl.solve_result}
    if ampl.solve_result == "solved":
        record["cost"] = ampl.var["TotalCost"].value()
        record["impact"] = ampl.var["TotalImpact"].value()
        record["gap"] = ampl.get_value("Cost.relmipgap")
    records.append(record)
    print(record)
front = sorted({(r["cost"], r["impact"]) for r in records if r["status"] == "solved"})
print("front:", front)

{'limit': 8, 'status': 'solved', 'cost': 1.0, 'impact': 8.0, 'gap': 0}
{'limit': 7, 'status': 'solved', 'cost': 4.0, 'impact': 6.0, 'gap': 0}
{'limit': 6, 'status': 'solved', 'cost': 4.0, 'impact': 6.0, 'gap': 0}
{'limit': 5, 'status': 'solved', 'cost': 8.0, 'impact': 1.0, 'gap': 0}
{'limit': 1, 'status': 'solved', 'cost': 8.0, 'impact': 1.0, 'gap': 0}
{'limit': 0, 'status': 'infeasible'}
front: [(1.0, 8.0), (4.0, 6.0), (8.0, 1.0)]


### Pure AMPL: data.dat and model.run

The same sweep in pure AMPL:

In [12]:
%%writefile data.dat
set OPTIONS := A B C D;
param: cost impact :=
    A   1   8
    B   4   6
    C   8   1
    D   5   7 ;
param cost_min := 1;
param cost_range := 7;
param impact_min := 1;
param impact_range := 7;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs, solver_msg 0;
objective Cost;
for {limit in {8, 7, 6, 5, 1, 0}} {
    let impact_limit := limit;
    solve;
    if solve_result = "solved" then
        printf "%g %s cost %g impact %g\n", limit, solve_result, TotalCost, TotalImpact;
    else
        printf "%g %s\n", limit, solve_result;
}
""")
except Exception:
    pass  # AMPL has printed the error above

8 solved cost 1 impact 8
7 solved cost 4 impact 6
6 solved cost 4 impact 6
5 solved cost 8 impact 1
1 solved cost 8 impact 1


[WARNING] 
	presolve, constraint ChooseOne:
		all variables eliminated, but lower bound = 1 > 0


0 infeasible
